# Morse audio to text

A walkthrough of the shared frontend, encoder, and CTC decoder. Install the repository with `pip install -e ".[dev]"` first. Signals below are synthetic; the randomly initialized model demonstrates tensor shapes, not transcription accuracy.

In [ ]:
from pathlib import Path
import torch
from IPython.display import Audio, Image, display
from morse_asr.demo import make_demo, synthesize
from morse_asr.audio import AudioConfig, features
from morse_asr.model import MorseRecognizer
from morse_asr.text import BLANK, encode, collapse_ctc

torch.manual_seed(42)
torch.set_num_threads(2)
output = Path("../runs/notebook-demo")
make_demo(output)

## 1. Inspect audio and mel features

The shared frontend resamples to mono 22,050 Hz and extracts 64 mel bands. Time hop: 256 samples; FFT window: 1,024 samples.

In [ ]:
display(Audio(filename=str(output / "sos-clean.wav")))
display(Image(filename=str(output / "mel-spectrograms.png")))
signal, _ = synthesize()
spec = features(signal, AudioConfig())
print("[channels, mel bins, frames]:", tuple(spec.shape))

## 2. Follow the model dimensions

Three convolutional blocks feed a five-layer bidirectional LSTM. No trained weights are bundled.

In [ ]:
model = MorseRecognizer().eval()
with torch.inference_mode():
    logits, valid_lengths = model(spec.unsqueeze(0), torch.tensor([spec.shape[-1]]))
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
print("[batch, output frames, classes]:", tuple(logits.shape))
print("Valid output lengths:", valid_lengths.tolist())

## 3. Decode CTC paths correctly

Repeated frame labels collapse, but a blank between two identical labels preserves two characters. The Cyrillic alphabet is part of the original task.

In [ ]:
a = encode("А")[0]
path = [a, a, BLANK, a, a]
assert collapse_ctc(path) == "АА"
print("Decoded path:", collapse_ctc(path))

## 4. Train and evaluate

Use the CLI commands in the README with explicit manifests. Prepare an independent holdout before training. The synthetic eight-recording set is for execution checks only. Read `docs/evaluation.md` for historical experiment provenance and metric definitions.